# Expectations about inflation, house prices, and the stock market

TECH2 term paper: an analysis of US household expectations using the Survey of Consumer Expectations (SCE), run by the Federal Reserve Bank of New York.

## Setup

In [1]:
import glob

import pandas as pd

## Part 1: Reading in the data

Each survey wave is stored in its own file named `SCE-MMM-YYYY.csv`. Each row is one respondent in that month.

### 1.1 Read and merge the SCE files

The files are semicolon-separated, so `sep=';'` is required. The pattern `SCE-*.csv` leaves out `CPI.csv`, which is in the same folder.

Filenames don't sort chronologically (`Apr-2014` comes before `Jun-2013`), so the merged data is sorted by date. Part 2 relies on this order to find each respondent's first observation.

In [2]:
files = glob.glob('data/SCE-*.csv')

# Read each survey wave and stack them into a single DataFrame
df_raw = pd.concat(
    [pd.read_csv(file, sep=';', parse_dates=['date']) for file in files],
    ignore_index=True,
)

# Put the observations in chronological order
df_raw = df_raw.sort_values(['date', 'userid'], ignore_index=True)

print(f'Read {len(files)} files into {df_raw.shape[1]} columns')

Read 139 files into 29 columns


### 1.2 Sample statistics

These statistics are reported again for the cleaned data in Part 2, so they are computed by a reusable function.

In [3]:
def sample_statistics(df):
    """
    Summarize the size and time span of a survey data set.

    Parameters
    ----------
    df : pandas.DataFrame
        Survey data with the columns ``userid``, ``wid`` and ``date``.

    Returns
    -------
    pandas.DataFrame
        Table with one row per statistic.
    """
    statistics = {
        'Unique individuals': f'{df["userid"].nunique():,}',
        'Observations': f'{len(df):,}',
        'Unique survey waves': f'{df["wid"].nunique():,}',
        'First date': f'{df["date"].min():%Y-%m-%d}',
        'Last date': f'{df["date"].max():%Y-%m-%d}',
    }

    return pd.DataFrame.from_dict(statistics, orient='index', columns=['Value'])

In [4]:
sample_statistics(df_raw)

,Value
Unique individuals,"23,369"
Observations,"176,101"
Unique survey waves,139
First date,2013-06-01
Last date,2024-12-31


The raw data has 176,101 observations from 23,369 individuals across 139 monthly waves, or about 7.5 observations per individual. That is fewer than the usual 12 because some respondents stop early, and those at the end of the sample haven't finished their 12 months.

The data starts in June 2013, before the 2014 start given in the task description.

## Part 2: Data pre-processing

This part fills in the numeracy answers, removes incomplete observations and outliers, and creates the grouping variables used in the rest of the analysis.

In [5]:
# Variables used throughout the analysis
DEMOGRAPHIC_COLS = ['female', 'age', 'educ']
EXPECTATION_COLS = ['inflation', 'house_price_change', 'prob_stocks_up']
NUMERACY_COLS = [f'num_lit_q{q}_correct' for q in (1, 2, 3, 5, 6, 8, 9)]

### 2.1 Fill in the numeracy answers

Numeracy questions are only asked the first time a respondent enters the survey. The data is sorted by date, so a forward fill within each respondent copies the first answers to all later observations.

In [6]:
df = df_raw.copy()

# Copy each respondent's first answers to their later observations
df[NUMERACY_COLS] = df.groupby('userid')[NUMERACY_COLS].ffill()

n_incomplete_before = df_raw[NUMERACY_COLS].isna().any(axis=1).sum()
n_incomplete_after = df[NUMERACY_COLS].isna().any(axis=1).sum()

print(f'Rows missing a numeracy answer before filling: {n_incomplete_before:,}')
print(f'Rows missing a numeracy answer after filling: {n_incomplete_after:,}')

Rows missing a numeracy answer before filling: 158,265
Rows missing a numeracy answer after filling: 37,378


### 2.2 Drop observations with missing values

The three groups of variables are checked in turn, so each count only includes rows that passed the previous steps.

In [7]:
missing_steps = {
    'Demographics': DEMOGRAPHIC_COLS,
    'Expectations': EXPECTATION_COLS,
    'Numeracy': NUMERACY_COLS,
}

missing_report = {}

for step, columns in missing_steps.items():
    n_before = len(df)
    df = df.dropna(subset=columns)
    missing_report[step] = {'Dropped': n_before - len(df), 'Remaining': len(df)}

pd.DataFrame.from_dict(missing_report, orient='index')

,Dropped,Remaining
Demographics,868,175233
Expectations,1683,173550
Numeracy,35974,137576


Most of the dropped rows are in the numeracy step. Questions 8 and 9 were not asked in 2013–14 and only partly in 2015, so requiring all seven answers removes the earliest respondents.

### 2.3 Drop outliers

Each expectations variable is processed in turn, using the data left after the previous variable. Both cutoffs for a variable are computed on the same sample.

In [8]:
outlier_report = {}

for column in EXPECTATION_COLS:
    lower, upper = df[column].quantile([0.001, 0.999])
    is_below = df[column] < lower
    is_above = df[column] > upper

    outlier_report[column] = {
        'Lower cutoff': lower,
        'Upper cutoff': upper,
        'Dropped below': is_below.sum(),
        'Dropped above': is_above.sum(),
    }

    df = df[~(is_below | is_above)]

print(f'Remaining observations: {len(df):,}')
pd.DataFrame.from_dict(outlier_report, orient='index')

Remaining observations: 137,278


,Lower cutoff,Upper cutoff,Dropped below,Dropped above
inflation,-75.0,100.0,92,27
house_price_change,-50.0,100.0,133,46
prob_stocks_up,0.0,100.0,0,0


No stock market responses are dropped. This variable is a probability, and its cutoffs equal the limits of 0 and 100, so no value can fall outside them.

### 2.4 College indicator

`educ` equals 4 for respondents with a bachelor's degree or higher.

In [9]:
df['college'] = (df['educ'] == 4).astype(int)

### 2.5 Numerical literacy

Numeracy answers don't change within a respondent, so the score is computed once per individual. Each individual counts once in the distribution and the median, however many times they were surveyed.

In [10]:
# Total correct answers for each individual, using one row per person
numeracy_score = df.groupby('userid')[NUMERACY_COLS].first().sum(axis=1).astype(int)

score_shares = numeracy_score.value_counts(normalize=True).sort_index()
median_score = numeracy_score.median()

print(f'Median number of correct answers: {median_score:.0f}')

# Display the share of individuals at each score as a percentage
score_shares.mul(100).round(1).rename('Share of individuals (%)').rename_axis(
    'Correct answers'
).to_frame()

Median number of correct answers: 6


,Share of individuals (%)
Correct answers,
0,0.1
1,0.7
2,2.4
3,5.7
4,10.5
5,16.9
6,27.4
7,36.2


In [11]:
# High numeracy means strictly more correct answers than the median
is_high_numeracy = numeracy_score > median_score
df['num_lit_high'] = df['userid'].map(is_high_numeracy).astype(int)

print(f'Share of individuals with high numeracy: {is_high_numeracy.mean():.1%}')
print(f'Share of observations with high numeracy: {df["num_lit_high"].mean():.1%}')

Share of individuals with high numeracy: 36.2%
Share of observations with high numeracy: 38.8%


The median respondent answered 6 of 7 questions correctly, so "high numeracy" means answering all seven correctly. This applies to 36.2% of individuals, so the two groups are unequal in size.

### 2.6 Sample statistics for the final data set

In [12]:
sample_statistics(df)

,Value
Unique individuals,"17,701"
Observations,"137,278"
Unique survey waves,117
First date,2015-04-02
Last date,2024-12-31


Cleaning leaves 137,278 observations from 17,701 individuals, about 78% of the original rows. The sample now starts in April 2015 because of the numeracy requirement, which matches the 2015–2024 period studied below.